In [ ]:
from time import sleep

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import pygame

In [29]:
learning_rate = 0.002
gamma         = 0.99
lmbda         = 0.95
eps_clip      = 0.2
K_epochs      = 4
T_horizon     = 20

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

class ActorCritic(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(ActorCritic, self).__init__()
        
        self.actor = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.Tanh(),
            nn.Linear(128, 64),
            nn.Tanh(),
            nn.Linear(64, action_dim),
            nn.Softmax(dim=-1)
        )
        
        self.critic = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.Tanh(),
            nn.Linear(128, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )

    def forward(self):
        raise NotImplementedError
    
    def act(self, state):
        state = torch.from_numpy(state).float().to(device)
        action_probs = self.actor(state)
        dist = Categorical(action_probs)
        action = dist.sample()
        action_logprob = dist.log_prob(action)
        return action.item(), action_logprob.item()
    
    def evaluate(self, state, action):
        action_probs = self.actor(state)
        dist = Categorical(action_probs)
        action_logprobs = dist.log_prob(action)
        dist_entropy = dist.entropy()
        state_values = self.critic(state)
        return action_logprobs, state_values, dist_entropy

class PPO:
    def __init__(self, state_dim, action_dim):
        self.policy = ActorCritic(state_dim, action_dim).to(device)
        self.optimizer = optim.Adam(self.policy.parameters(), lr=learning_rate)
        self.policy_old = ActorCritic(state_dim, action_dim).to(device)
        self.policy_old.load_state_dict(self.policy.state_dict())
        self.mse_loss = nn.MSELoss()
        self.data = []
        
    def put_data(self, item):
        self.data.append(item)
        
    def make_batch(self):
        s_lst, a_lst, r_lst, s_prime_lst, prob_a_lst, done_lst = [], [], [], [], [], []
        for transition in self.data:
            s, a, r, s_prime, prob_a, done = transition
            s_lst.append(s)
            a_lst.append([a])
            r_lst.append([r])
            s_prime_lst.append(s_prime)
            prob_a_lst.append([prob_a])
            done_mask = 0 if done else 1
            done_lst.append([done_mask])
            
        s,a,r,s_prime,done_mask, prob_a = torch.tensor(np.array(s_lst), dtype=torch.float).to(device), \
                                          torch.tensor(np.array(a_lst), dtype=torch.float).to(device), \
                                          torch.tensor(np.array(r_lst), dtype=torch.float).to(device), \
                                          torch.tensor(np.array(s_prime_lst), dtype=torch.float).to(device), \
                                          torch.tensor(np.array(done_lst), dtype=torch.float).to(device), \
                                          torch.tensor(np.array(prob_a_lst), dtype=torch.float).to(device)
        self.data = []
        return s, a, r, s_prime, done_mask, prob_a

    def update(self):
        s, a, r, s_prime, done_mask, old_log_prob = self.make_batch()

        with torch.no_grad():
            target = r + gamma * self.policy_old.critic(s_prime) * done_mask
            advantage = target - self.policy_old.critic(s)
            advantage = (advantage - advantage.mean()) / (advantage.std() + 1e-10)

        for _ in range(K_epochs):
            log_probs, state_values, dist_entropy = self.policy.evaluate(s, a.squeeze())
            ratios = torch.exp(log_probs - old_log_prob.squeeze())
            surr1 = ratios * advantage.squeeze()
            surr2 = torch.clamp(ratios, 1-eps_clip, 1+eps_clip) * advantage.squeeze()
            loss = -torch.min(surr1, surr2) + 0.5 * self.mse_loss(state_values, target) - 0.01 * dist_entropy

            self.optimizer.zero_grad()
            loss.mean().backward()
            self.optimizer.step()
            
        self.policy_old.load_state_dict(self.policy.state_dict())

def main():
    env = gym.make('CartPole-v1', render_mode=None) 
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n
    model = PPO(state_dim, action_dim)
    
    score = 0.0
    print_interval = 20
    
    for n_epi in range(1000):
        s, _ = env.reset()
        done = False
        truncated = False
        
        while not done and not truncated:
            for t in range(T_horizon):
                prob = model.policy_old.act(s)
                a = prob[0]
                a_log_prob = prob[1]
                s_prime, r, done, truncated, info = env.step(a)
                model.put_data((s, a, r/100.0, s_prime, a_log_prob, done or truncated))
                s = s_prime
                score += r
                if done or truncated:
                    break
            model.update()

        if n_epi % print_interval == 0 and n_epi != 0:
            print(f"Épisode: {n_epi}, Score moyen: {score/print_interval:.1f}")
            if score/print_interval > 450:
                print(f"Environnement résolu ! Score final : {score/print_interval}")
                break
            score = 0.0

    env.close()
    
    env_show = gym.make('CartPole-v1', render_mode=None)
    s, _ = env_show.reset()
    done = False
    truncated = False
    total_reward = 0
    while not done and not truncated:
        a, _ = model.policy_old.act(s)
        s, r, done, truncated, _ = env_show.step(a)
        total_reward += r
    print(f"Score de la démonstration : {total_reward}")
    env_show.close()

if __name__ == '__main__':
    main()

Épisode: 20, Score moyen: 33.0


/tmp/ipykernel_648/56142948.py:89: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1857.)
  advantage = (advantage - advantage.mean()) / (advantage.std() + 1e-10)


ValueError: Expected parameter probs (Tensor of shape (1, 2)) of distribution Categorical(probs: torch.Size([1, 2])) to satisfy the constraint Simplex(), but found invalid values:
tensor([[nan, nan]], device='cuda:0', grad_fn=<DivBackward0>)